In [ ]:
# ============================================================
# 1) Check GPU
# ============================================================

# Check GPU availability and CUDA support before
# running the Stable Diffusion and LoRA training pipeline.

!nvidia-smi

In [ ]:
# ============================================================
# 2) Safe install
# ============================================================

# Prepare the Colab environment and install the required
# libraries for Stable Diffusion, LoRA fine-tuning,
# dataset handling, and image processing.

%cd /content

# Remove old folders to avoid conflicts
!rm -rf /content/hf_diffusers
!rm -rf /content/diffusers

# Update pip without changing the default torch version
!python -m pip install -q -U pip

# Install the required project libraries
!python -m pip install -q -U \
  diffusers \
  transformers \
  accelerate \
  datasets \
  peft \
  safetensors \
  kagglehub \
  matplotlib \
  "pillow<12" \
  "numpy>=1.26,<2.1"

# Remove torchao because it conflicts with peft
!python -m pip uninstall -y torchao -q

# Clone the official Hugging Face Diffusers repository
# to access the LoRA training script
!git clone -q https://github.com/huggingface/diffusers /content/hf_diffusers

print("Install finished.")

In [ ]:
# ============================================================
# 3) Imports
# ============================================================

# Import the required libraries for dataset processing,
# image handling, visualization, and Stable Diffusion generation.

import os
import re
import json
import math
import random
import shutil
import importlib.util
import subprocess
from pathlib import Path
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageFilter, ImageEnhance

import torch
import diffusers
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

# Display library versions and CUDA availability
print("torch:", torch.__version__)
print("diffusers:", diffusers.__version__)
print("CUDA:", torch.cuda.is_available())
print("torchao installed?", importlib.util.find_spec("torchao") is not None)

# Configure the device and data type for execution
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

In [ ]:
# ============================================================
# 4) Google Drive folders
# ============================================================

# Connect Google Drive to save project files and outputs
from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/AgriGen_Lite_Fast"

# Define the main project folders
TRAIN_DATA_DIR = "/content/agrigen_lite_training_dataset"
LORA_OUTPUT_DIR = os.path.join(PROJECT_DIR, "lora_weights")
RESULTS_DIR = os.path.join(PROJECT_DIR, "generated_images")
METADATA_DIR = os.path.join(PROJECT_DIR, "metadata")
ZIP_DIR = os.path.join(PROJECT_DIR, "zips")
TRAIN_LOG_PATH = os.path.join(PROJECT_DIR, "training_log.txt")

# Create folders if they do not already exist
for d in [LORA_OUTPUT_DIR, RESULTS_DIR, METADATA_DIR, ZIP_DIR]:
    os.makedirs(d, exist_ok=True)

print("Project folder:", PROJECT_DIR)
print("LoRA output:", LORA_OUTPUT_DIR)

In [ ]:
# ============================================================
# 5) Download Fruits-360
# ============================================================

# Download the Fruits-360 dataset using KaggleHub
import kagglehub

dataset_path = kagglehub.dataset_download("moltean/fruits")
print("Dataset path:", dataset_path)

# Locate the main dataset folder containing
# the Training and Test directories
def find_dataset_root(base_path):
    candidates = []
    for root, dirs, files in os.walk(base_path):
        if "Training" in dirs and "Test" in dirs:
            train_dir = os.path.join(root, "Training")

            # Count the available training classes
            n_classes = len([
                d for d in os.listdir(train_dir)
                if os.path.isdir(os.path.join(train_dir, d))
            ])

            candidates.append((n_classes, root))

    if not candidates:
        raise FileNotFoundError("Could not find Training/Test folders.")

    candidates = sorted(candidates, key=lambda x: x[0], reverse=True)
    return candidates[0][1]

DATA_ROOT = find_dataset_root(dataset_path)
TRAIN_DIR = os.path.join(DATA_ROOT, "Training")

print("TRAIN_DIR:", TRAIN_DIR)
print("Classes:", len(os.listdir(TRAIN_DIR)))

In [ ]:
# ============================================================
# 6) Settings
# ============================================================

# Define the main training settings for Stable Diffusion
# and LoRA fine-tuning.

MODEL_ID = "runwayml/stable-diffusion-v1-5"
TRIGGER_WORD = "agrigenstyle"

# Training configuration
RESOLUTION = 384
IMAGES_PER_CLASS = 6
MAX_TRAIN_STEPS = 600
LORA_RANK = 4

TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 1
LEARNING_RATE = 1e-4
MIXED_PRECISION = "no"

print("RESOLUTION:", RESOLUTION)
print("IMAGES_PER_CLASS:", IMAGES_PER_CLASS)
print("MAX_TRAIN_STEPS:", MAX_TRAIN_STEPS)
print("LORA_RANK:", LORA_RANK)

In [ ]:
# ============================================================
# 7) Prepare balanced dataset from all classes
# ============================================================

# Clean and standardize class names from the dataset folders
def normalize_class_name(name):
    name = str(name).lower().strip()
    name = name.replace("_", " ").replace("-", " ")
    name = re.sub(r"\d+", "", name)
    name = re.sub(r"\s+", " ", name)
    return name.strip()

# Convert technical class names into clearer prompt objects
def class_to_prompt_object(class_name):
    name = normalize_class_name(class_name)

    replacements = {
        "tomatoe": "tomato",
        "apple granny smith": "green apple",
        "apple golden": "golden apple",
        "apple red": "red apple",
        "apple crimson snow": "red apple",
        "apple braeburn": "apple",
        "banana red": "red banana",
        "cherry rainier": "rainier cherry",
        "pepper red": "red pepper",
        "pepper green": "green pepper",
        "pepper yellow": "yellow pepper",
        "potato red": "red potato",
        "potato white": "white potato",
        "grape blue": "blue grape",
        "pear red": "red pear",
    }

    # Apply manual replacements for clearer text prompts
    for k, v in replacements.items():
        if k in name:
            return v

    return name.replace(" not ripened", " unripe").strip()

# Prepare a smaller balanced dataset for faster LoRA training
def prepare_lite_dataset(train_dir, output_dir, images_per_class=6):
    output_dir = Path(output_dir)

    # Recreate the output folder to avoid mixing old and new files
    if output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    valid_ext = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    # Get all class folders from the training directory
    class_names = sorted([
        d for d in os.listdir(train_dir)
        if os.path.isdir(os.path.join(train_dir, d))
    ])

    metadata = []
    class_mapping = {}

    # Use a fixed seed to make image selection reproducible
    random.seed(42)

    for class_name in class_names:
        class_dir = Path(train_dir) / class_name

        # Collect valid image files for the current class
        image_paths = [
            p for p in class_dir.iterdir()
            if p.is_file() and p.suffix.lower() in valid_ext
        ]

        # Randomly select a fixed number of images from each class
        random.shuffle(image_paths)
        image_paths = image_paths[:images_per_class]

        # Convert the folder name into a prompt-friendly object name
        prompt_object = class_to_prompt_object(class_name)
        class_mapping[class_name] = prompt_object

        safe_class = re.sub(r"[^a-zA-Z0-9]+", "_", class_name).strip("_")

        for i, src in enumerate(image_paths):
            file_name = f"{safe_class}_{i:03d}.jpg"
            dst = output_dir / file_name

            # Convert images to RGB and resize them for training
            img = Image.open(src).convert("RGB")
            img = img.resize((RESOLUTION, RESOLUTION), Image.Resampling.LANCZOS)
            img.save(dst, quality=95)

            # Create a caption linked to the image for text-to-image training
            caption = (
                f"a realistic close-up photo of {TRIGGER_WORD} {prompt_object} "
                f"on a plain white background, centered composition, sharp details, studio lighting"
            )

            metadata.append({"file_name": file_name, "text": caption})

    # Save the unique supported prompt objects for later inference
    supported_prompts = sorted(list(set(class_mapping.values())))

    # Save image-caption pairs in metadata.jsonl format
    with open(output_dir / "metadata.jsonl", "w", encoding="utf-8") as f:
        for row in metadata:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    # Save the mapping between original class names and prompt names
    with open(os.path.join(METADATA_DIR, "class_mapping.json"), "w", encoding="utf-8") as f:
        json.dump(class_mapping, f, ensure_ascii=False, indent=2)

    # Save the list of supported prompt objects
    with open(os.path.join(METADATA_DIR, "supported_prompts.json"), "w", encoding="utf-8") as f:
        json.dump(supported_prompts, f, ensure_ascii=False, indent=2)

    print("Prepared images:", len(metadata))
    print("Classes used:", len(class_names))
    print("Supported prompt objects:", len(supported_prompts))

    return metadata, class_mapping, supported_prompts

# Run the dataset preparation function
metadata, class_mapping, supported_prompts = prepare_lite_dataset(
    TRAIN_DIR,
    TRAIN_DATA_DIR,
    images_per_class=IMAGES_PER_CLASS
)

# Print example captions to verify the generated metadata
print("\nFirst captions:")
for row in metadata[:5]:
    print(row["text"])

In [ ]:
# ============================================================
# 8) Train LoRA fast
# ============================================================

# Remove previous LoRA outputs before starting a new training run
if os.path.exists(LORA_OUTPUT_DIR):
    shutil.rmtree(LORA_OUTPUT_DIR)
os.makedirs(LORA_OUTPUT_DIR, exist_ok=True)

# Use the official Hugging Face Diffusers LoRA training script
train_script = "/content/hf_diffusers/examples/text_to_image/train_text_to_image_lora.py"

# Define the LoRA training command and its main hyperparameters
cmd = [
    "accelerate", "launch",
    f"--mixed_precision={MIXED_PRECISION}",
    train_script,
    "--pretrained_model_name_or_path", MODEL_ID,
    "--train_data_dir", TRAIN_DATA_DIR,
    "--image_column", "image",
    "--caption_column", "text",
    "--resolution", str(RESOLUTION),
    "--center_crop",
    "--random_flip",
    "--train_batch_size", str(TRAIN_BATCH_SIZE),
    "--gradient_accumulation_steps", str(GRADIENT_ACCUMULATION_STEPS),
    "--max_train_steps", str(MAX_TRAIN_STEPS),
    "--learning_rate", str(LEARNING_RATE),
    "--lr_scheduler", "cosine",
    "--lr_warmup_steps", "0",
    "--rank", str(LORA_RANK),
    "--gradient_checkpointing",
    "--checkpointing_steps", "300",
    "--checkpoints_total_limit", "1",
    "--output_dir", LORA_OUTPUT_DIR,
]

# Print the full command for verification
print(" ".join(cmd))

# Run the training process and save the output logs
with open(TRAIN_LOG_PATH, "w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    # Display and store training progress line by line
    for line in process.stdout:
        print(line, end="")
        log_file.write(line)

    exit_code = process.wait()

print("Exit code:", exit_code)

# Show the last log lines if training fails
if exit_code != 0:
    print("\nTraining failed. Last 80 lines:")
    !tail -n 80 "{TRAIN_LOG_PATH}"
else:
    print("\nTraining finished.")

In [ ]:
# ============================================================
# 9) Find LoRA weights
# ============================================================

# Search for the generated LoRA weight files after training
def find_lora_weights(lora_dir):
    lora_dir = Path(lora_dir)

    # Possible LoRA weight file names
    candidate_names = [
        "pytorch_lora_weights.safetensors",
        "pytorch_lora_weights.bin",
        "adapter_model.safetensors",
        "adapter_model.bin",
    ]

    # Check the main output folder first
    for name in candidate_names:
        p = lora_dir / name
        if p.exists():
            return p.parent, p.name

    # Search recursively inside subfolders if needed
    for name in candidate_names:
        matches = sorted(lora_dir.rglob(name))
        if matches:
            p = matches[-1]
            return p.parent, p.name

    # Display existing files if no LoRA weights are found
    print("Files found:")
    !find "{LORA_OUTPUT_DIR}" -maxdepth 4 -type f | sed -n '1,200p'

    raise FileNotFoundError("No LoRA weights found. Check training_log.txt")

# Locate the trained LoRA weights
lora_folder, lora_weight_name = find_lora_weights(LORA_OUTPUT_DIR)

print("LoRA found:")
print("Folder:", lora_folder)
print("File:", lora_weight_name)

In [ ]:
# ============================================================
# 10) Load model
# ============================================================

# Load the Stable Diffusion v1.5 pipeline
pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    use_safetensors=True
)

# Configure the scheduler and optimization settings
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(device)
pipe.enable_attention_slicing()
pipe.enable_vae_slicing()

# Load the trained LoRA weights into the pipeline
pipe.load_lora_weights(str(lora_folder), weight_name=lora_weight_name)

print("Loaded SD1.5 + Lite LoRA.")

In [ ]:
# ============================================================
# 11) Prompt + styles
# ============================================================

# Define common agricultural objects as a fallback list
COMMON_OBJECTS = {
    "apple", "banana", "cherry", "tomato", "orange", "lemon", "strawberry", "mango",
    "pear", "peach", "kiwi", "pineapple", "watermelon", "grape", "pomegranate",
    "avocado", "cucumber", "carrot", "potato", "onion", "pepper", "corn", "eggplant",
    "plum", "apricot", "cantaloupe", "raspberry", "blueberry", "cabbage", "cauliflower",
    "broccoli", "walnut", "hazelnut", "almond", "pistachio", "peanut"
}

# Clean user text before matching it with supported prompts
def clean_text(text):
    text = str(text).lower().strip()
    text = text.replace("_", " ").replace("-", " ")
    text = re.sub(r"[^a-z0-9 ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Identify the main object from the user input
def resolve_object(text):
    text = clean_text(text)

    # Check exact match with supported prompts
    if text in supported_prompts:
        return text

    # Check partial matches with supported prompts
    for obj in sorted(supported_prompts, key=len, reverse=True):
        if text == obj or text in obj or obj in text:
            return obj

    # Fall back to common agricultural objects
    for obj in sorted(COMMON_OBJECTS, key=len, reverse=True):
        if re.search(r"\b" + re.escape(obj) + r"\b", text):
            return obj

    # Return cleaned text if no predefined object is matched
    return text

# Build the positive and negative prompts for image generation
def build_prompt(text):
    obj = resolve_object(text)

    prompt = (
        f"a realistic close-up photo of {TRIGGER_WORD} {obj} "
        f"on a plain white background, centered composition, sharp details, studio lighting, high quality"
    )

    negative = (
        "blurry, low quality, cartoon, drawing, painting, illustration, distorted, deformed, "
        "multiple fruits, duplicate, extra objects, text, watermark, logo, hands, people, noisy background"
    )

    return prompt, negative, obj

# Apply optional post-processing styles to the generated image
def apply_style(image, style="none"):
    style = str(style).lower().strip()
    img = image.convert("RGB")

    # Return the original image without changes
    if style in ["none", "normal", "original", ""]:
        return img

    # Convert the image to grayscale with contrast enhancement
    if style in ["gray", "grey", "grayscale"]:
        gray = ImageOps.grayscale(img)
        gray = ImageOps.autocontrast(gray)
        return gray.convert("RGB")

    # Create a pencil sketch effect
    if style in ["sketch", "pencil", "drawing"]:
        gray = ImageOps.grayscale(img)
        inverted = ImageOps.invert(gray)
        blurred = inverted.filter(ImageFilter.GaussianBlur(radius=12))

        gray_np = np.array(gray).astype(np.float32)
        blur_np = np.array(blurred).astype(np.float32)

        sketch_np = gray_np * 255.0 / (255.0 - blur_np + 1e-6)
        sketch_np = np.clip(sketch_np, 0, 255).astype(np.uint8)

        sketch = Image.fromarray(sketch_np)
        sketch = ImageOps.autocontrast(sketch)
        return sketch.convert("RGB")

    # Enhance brightness, contrast, sharpness, and color
    if style in ["lighting", "light", "enhance"]:
        img = ImageEnhance.Brightness(img).enhance(1.12)
        img = ImageEnhance.Contrast(img).enhance(1.12)
        img = ImageEnhance.Sharpness(img).enhance(1.35)
        img = ImageEnhance.Color(img).enhance(1.05)
        return img.convert("RGB")

    raise ValueError("Style must be: none, gray, sketch, lighting")

In [ ]:
# ============================================================
# 12) Generate
# ============================================================

# Display generated images with optional titles
def show_images(images, titles=None):
    n = len(images)
    plt.figure(figsize=(5 * n, 5))

    for i, img in enumerate(images):
        plt.subplot(1, n, i + 1)
        plt.imshow(img)
        plt.axis("off")
        if titles:
            plt.title(titles[i], fontsize=10, fontweight="bold")

    plt.tight_layout()
    plt.show()

# Generate an image from a text prompt using the LoRA-enhanced pipeline
@torch.no_grad()
def generate_image(text, style="none", seed=42, steps=25, guidance_scale=7.5):
    # Build the positive and negative prompts
    prompt, negative, obj = build_prompt(text)

    print("Input:", text)
    print("Object:", obj)
    print("Style:", style)

    # Use a fixed seed for reproducible generation
    generator = torch.Generator(device=device).manual_seed(int(seed))

    # Generate the image using Stable Diffusion
    image = pipe(
        prompt=prompt,
        negative_prompt=negative,
        num_inference_steps=steps,
        guidance_scale=guidance_scale,
        width=512,
        height=512,
        generator=generator
    ).images[0]

    # Apply the selected post-processing style
    image = apply_style(image, style=style)

    # Create an output folder based on object name and style
    out_dir = os.path.join(RESULTS_DIR, obj.replace(" ", "_"), style)
    os.makedirs(out_dir, exist_ok=True)

    # Save the generated image with a timestamped filename
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename_obj = obj.replace(" ", "_")
    out_path = os.path.join(out_dir, f"{filename_obj}_{style}_{timestamp}.png")
    image.save(out_path)

    print("Saved:", out_path)
    show_images([image], [f"{obj} | {style}"])

    return image

# Generate sample outputs with different objects and styles
generate_image("banana", style="none", seed=1)
generate_image("apple", style="lighting", seed=2)
generate_image("mango", style="gray", seed=3)
generate_image("tomato", style="sketch", seed=4)

In [ ]:
# ============================================================
# Evaluation Matrix using CLIP
# ============================================================

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

# ------------------------------------------------------------
# 1) Evaluation classes
# ------------------------------------------------------------

# Define the agricultural classes used for evaluation
EVAL_CLASSES = [
    "apple",
    "banana",
    "mango",
    "tomato",
    "orange",
    "strawberry"
]

IMAGES_PER_CLASS_EVAL = 3
EVAL_STEPS = 25
GUIDANCE_SCALE = 7.5

# ------------------------------------------------------------
# 2) Load CLIP model for image classification
# ------------------------------------------------------------

# Load the CLIP model and processor for zero-shot evaluation
clip_model_id = "openai/clip-vit-base-patch32"

clip_model = CLIPModel.from_pretrained(clip_model_id).to(device)
clip_processor = CLIPProcessor.from_pretrained(clip_model_id)

clip_model.eval()

print("CLIP model loaded.")

# ------------------------------------------------------------
# 3) Helper: CLIP prediction
# ------------------------------------------------------------

@torch.no_grad()
def predict_image_class_with_clip(image, class_names):
    """
    Predict image class using CLIP zero-shot classification.
    """

    # Create text prompts for all evaluation classes
    text_prompts = [
        f"a photo of a {name} fruit" for name in class_names
    ]

    # Process the image and text prompts for CLIP inference
    inputs = clip_processor(
        text=text_prompts,
        images=image,
        return_tensors="pt",
        padding=True
    ).to(device)

    outputs = clip_model(**inputs)

    # Compute similarity probabilities between image and text prompts
    logits = outputs.logits_per_image
    probs = logits.softmax(dim=1)

    pred_idx = probs.argmax(dim=1).item()
    confidence = probs[0, pred_idx].item()

    return pred_idx, confidence, probs.cpu().numpy()[0]

# ------------------------------------------------------------
# 4) Generate images and evaluate
# ------------------------------------------------------------

true_labels = []
pred_labels = []
all_results = []

# Generate evaluation samples for each agricultural class
for true_idx, fruit_name in enumerate(EVAL_CLASSES):
    for i in range(IMAGES_PER_CLASS_EVAL):

        prompt, negative, obj = build_prompt(fruit_name)

        # Use a fixed seed for reproducible evaluation results
        generator = torch.Generator(device=device).manual_seed(1000 + true_idx * 10 + i)

        # Generate an image using Stable Diffusion and LoRA
        image = pipe(
            prompt=prompt,
            negative_prompt=negative,
            num_inference_steps=EVAL_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            width=512,
            height=512,
            generator=generator
        ).images[0]

        # Predict the generated image class using CLIP
        pred_idx, confidence, probs = predict_image_class_with_clip(
            image,
            EVAL_CLASSES
        )

        true_labels.append(true_idx)
        pred_labels.append(pred_idx)

        all_results.append({
            "true": fruit_name,
            "pred": EVAL_CLASSES[pred_idx],
            "confidence": confidence,
            "image": image
        })

        print(
            f"True: {fruit_name:12s} | "
            f"Pred: {EVAL_CLASSES[pred_idx]:12s} | "
            f"Conf: {confidence:.3f}"
        )

# ------------------------------------------------------------
# 5) Build confusion matrix manually
# ------------------------------------------------------------

num_classes = len(EVAL_CLASSES)

# Create an empty confusion matrix
conf_matrix = np.zeros((num_classes, num_classes), dtype=int)

# Count prediction results for each class
for t, p in zip(true_labels, pred_labels):
    conf_matrix[t, p] += 1

# Calculate overall evaluation accuracy
accuracy = np.trace(conf_matrix) / np.sum(conf_matrix)

print("\nEvaluation Accuracy:", round(accuracy * 100, 2), "%")
print("Confusion Matrix:")
print(conf_matrix)

# ------------------------------------------------------------
# 6) Plot confusion matrix
# ------------------------------------------------------------

# Visualize the confusion matrix results
plt.figure(figsize=(9, 7))
plt.imshow(conf_matrix, interpolation="nearest")

plt.title(f"Evaluation Confusion Matrix | Accuracy = {accuracy*100:.2f}%")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")

plt.xticks(np.arange(num_classes), EVAL_CLASSES, rotation=45, ha="right")
plt.yticks(np.arange(num_classes), EVAL_CLASSES)

# Display the prediction counts inside the matrix cells
for i in range(num_classes):
    for j in range(num_classes):
        plt.text(
            j,
            i,
            str(conf_matrix[i, j]),
            ha="center",
            va="center",
            fontsize=12,
            fontweight="bold"
        )

plt.colorbar()
plt.tight_layout()

# Save the confusion matrix figure
eval_matrix_path = os.path.join(RESULTS_DIR, "evaluation_confusion_matrix.png")
plt.savefig(eval_matrix_path, dpi=150, bbox_inches="tight")

plt.show()

print("Saved evaluation matrix to:", eval_matrix_path)

In [ ]:
# ============================================================
# Show evaluation samples
# ============================================================

# Select up to 12 generated evaluation samples to display
n_show = min(12, len(all_results))

plt.figure(figsize=(12, 8))

# Display each sample with its true label, predicted label, and confidence score
for i in range(n_show):
    item = all_results[i]
    img = item["image"]

    plt.subplot(3, 4, i + 1)
    plt.imshow(img)
    plt.axis("off")
    plt.title(
        f"True: {item['true']}\nPred: {item['pred']}\nConf: {item['confidence']:.2f}",
        fontsize=9
    )

plt.tight_layout()

# Save the displayed evaluation samples as an image file
samples_path = os.path.join(RESULTS_DIR, "evaluation_samples.png")
plt.savefig(samples_path, dpi=150, bbox_inches="tight")

plt.show()

print("Saved samples to:", samples_path)

In [ ]:
# ============================================================
# 13) Save zip files in Drive
# ============================================================

# Define the output ZIP file paths
zip_lora = os.path.join(ZIP_DIR, "lora_weights")
zip_results = os.path.join(ZIP_DIR, "generated_images")
zip_metadata = os.path.join(ZIP_DIR, "metadata")

# Remove old ZIP files if they already exist
for p in [zip_lora + ".zip", zip_results + ".zip", zip_metadata + ".zip"]:
    if os.path.exists(p):
        os.remove(p)

# Compress the LoRA weights, generated images, and metadata folders
shutil.make_archive(zip_lora, "zip", LORA_OUTPUT_DIR)
shutil.make_archive(zip_results, "zip", RESULTS_DIR)
shutil.make_archive(zip_metadata, "zip", METADATA_DIR)

print("Saved:")
print(zip_lora + ".zip")
print(zip_results + ".zip")
print(zip_metadata + ".zip")